# Structured Data Extraction with Nugen

This cookbook demonstrates how to reliably extract structured data (JSON) from unstructured text using Nugen models.

**Use Cases:**
- converting invoices to accounting entries
- parsing resumes into candidate profiles
- extracting action items from meeting notes

## 1. Setup
install necessary packages and setup the environment.

In [ ]:
%pip install requests python-dotenv

import os
import json
import requests
from dotenv import load_dotenv

# Load API Key
load_dotenv()
NUGEN_API_KEY = os.getenv("NUGEN_API_KEY")

if not NUGEN_API_KEY:
    raise ValueError("Please set NUGEN_API_KEY in your .env file")

# Configuration
API_URL = "https://api.nugen.in/inference/chat/completions"
MODEL = "nugen-flash-instruct"

## 2. Define Extraction Helper
We create a reusable function that wraps the Nugen API call. It enforces a system prompt that mandates JSON output.

In [ ]:
def extract_data(text, schema_desc):
    """
    Extracts data from text based on a schema description.
    """
    
    system_prompt = (
        "You are a precision data extraction engine. "
        "Your goal is to extract specific information from the provided text "
        "and output it as valid, parseable JSON only. "
        "Do not include markdown formatting (like ```json), explanations, or chatter."
    )
    
    user_prompt = f"""Source Text:
    {text}
    
    Required JSON Schema/Fields:
    {schema_desc}
    
    JSON Output:"""
    
    headers = {
        "Authorization": f"Bearer {NUGEN_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1  # Low temperature for deterministic output
    }
    
    try:
        response = requests.post(API_URL, json=payload, headers=headers)
        response.raise_for_status()
        result = response.json()
        
        content = result['choices'][0]['message']['content'].strip()
        
        # Cleanups common markdown artifacts if model slips up
        if content.startswith("```json"):
            content = content[7:]
        if content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
            
        return json.loads(content)
        
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw output:")
        print(content)
        return None
    except Exception as e:
        print(f"API Request failed: {e}")
        return None

## 3. Example: Invoice Processing
Let's try to extract invoice details including line items.

In [ ]:
invoice_text = """
INVOICE #1023
Date: Jan 28, 2025
To: Acme Corp

Items:
1. Cloud Hosting - Pro Plan ... $299.00
2. Domain Renewal (1 year) ... $15.00
3. Support Add-on ... $50.00

Subtotal: $364.00
Tax (10%): $36.40
Total Due: $400.40
"""

schema = """
{
  "invoice_number": "string",
  "date": "ISO8601 date string",
  "customer_name": "string",
  "line_items": [
    {"description": "string", "amount": "float"}
  ],
  "total_due": "float"
}
"""

data = extract_data(invoice_text, schema)
print(json.dumps(data, indent=2))

## 4. Troubleshooting

### Common Issues
1. **Invalid JSON**: Sometimes models add conversational text. The `extract_data` function above handles basic cleanup (removing markdown code blocks).
2. **Hallucinations**: Ensure your temperature is set low (0.0 - 0.2) to strictly adhere to the source text.
3. **Rate Limits**: If processing many files, implement backoff/retry logic (see `requests` adapters or `tenacity` library).

### Verification
Always validate the output schema in production using libraries like `pydantic` or `jsonschema`.